# 05 - Model Development & Evaluation

**Prerequisites:** Notebooks 01-04 completed, feature store populated.

**Purpose:** Train, evaluate, and compare models (literature-aligned progression):
1. Majority class (floor)
2. Logistic Regression (minimum viable - Zeleznikow 2023)
3. Random Forest (Katz 2017: 70.2% on SCOTUS; JES 2024: confirmed top-tier)
4. XGBoost tabular-only (primary candidate - JES 2024: 72% best performer)
5. XGBoost tabular + text (only if it improves over #4 - Aletras 2016: text adds ~3-5pp)

**Evaluation Gates (literature-aligned):**
- Test accuracy >= 70% (Katz 70.2%, Aletras 79%, JES 2024 72%)
- AUC-ROC >= 0.70 (PILOT 2024: 0.83 on ECHR)
- Brier score <= 0.22 (standard for well-calibrated models)
- Fairness delta < 5pp

**Key features include lawyer/advocate data** (LexEdge 2024, Pre/Dicta: counsel is a top predictor)

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import mlflow

from configs.settings import settings
from src.features.feature_store import FeatureStore
from src.models.baseline import train_baselines
from src.models.random_forest_model import train_random_forest
from src.models.xgboost_model import train_xgboost
from src.models.calibration import CalibratedModel, reliability_diagram_data
from src.models.evaluation import evaluate_model, check_gates, compute_shap_explanations

%matplotlib inline

## 1. Load Features and Split

In [ ]:
store = FeatureStore()
# df = store.load_features('v1')
# train_df, val_df, test_df = store.temporal_split(df)
# print(f'Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}')

## 2. Baseline Models

In [ ]:
# baseline_results = train_baselines(train_df, val_df, test_df)
# for name, metrics in baseline_results.items():
#     print(f'\n{name}:')
#     for k, v in metrics.items():
#         print(f'  {k}: {v:.4f}')

## 3. Random Forest (Katz 2017, JES 2024)

Katz et al. (2017) achieved 70.2% on SCOTUS with Random Forest. JES 2024 independently confirmed RF as competitive for legal prediction. RF handles mixed features well and provides strong feature importance interpretability — critical for lawyer trust.

In [ ]:
# rf_model, rf_metrics = train_random_forest(
#     train_df, val_df, test_df,
#     n_trials=50,  # Fewer trials than XGBoost since RF is less sensitive to HPO
# )
# print('Random Forest:', rf_metrics)

In [ ]:
## 4. XGBoost (Tabular Only)

JES 2024: XGBoost achieved 72% accuracy on SCOTUS, outperforming all other algorithms tested (NB 61%, DT 52%). This is our primary candidate.

## 4. XGBoost (Tabular + Text Embeddings)

**Only proceed if tabular-only model shows promise (AUC > 0.60).**

In [ ]:
# if metrics_tab.get('test_auc_roc', 0) > 0.60:
#     model_text, metrics_text = train_xgboost(
#         train_df, val_df, test_df,
#         include_text=True,
#         n_trials=100,
#     )
#     print('XGBoost (tabular + text):', metrics_text)
#     
#     # Compare: does text add value?
#     delta_auc = metrics_text['test_auc_roc'] - metrics_tab['test_auc_roc']
#     print(f'\nAUC improvement from text: {delta_auc:+.4f}')
#     if delta_auc > 0.01:
#         print('Text features ADD value. Use combined model.')
#         best_model = model_text
#         best_metrics = metrics_text
#     else:
#         print('Text features do NOT add value. Use tabular-only model.')
#         best_model = model_tab
#         best_metrics = metrics_tab
# else:
#     print('Tabular model AUC too low. Skipping text features.')
#     best_model = model_tab
#     best_metrics = metrics_tab

## 5. Calibration

In [ ]:
# Apply Platt scaling on validation set
# import xgboost as xgb
# import numpy as np
# 
# feature_cols = [c for c in val_df.columns if c != 'outcome_binary']
# dval = xgb.DMatrix(val_df[feature_cols].values, feature_names=feature_cols)
# val_probs = best_model.predict(dval)
# val_labels = val_df['outcome_binary'].values
# 
# calibrator = CalibratedModel(method='platt')
# calibrator.fit(val_probs, val_labels)
# 
# # Reliability diagram
# dtest = xgb.DMatrix(test_df[feature_cols].values, feature_names=feature_cols)
# test_probs_raw = best_model.predict(dtest)
# test_probs_cal = calibrator.predict_proba(test_probs_raw)
# 
# for label, probs in [('Raw', test_probs_raw), ('Calibrated', test_probs_cal)]:
#     data = reliability_diagram_data(test_df['outcome_binary'].values, probs)
#     plt.plot(data['bin_centers'], data['true_frequencies'], 'o-', label=label)
# plt.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
# plt.xlabel('Predicted probability')
# plt.ylabel('True frequency')
# plt.title('Reliability Diagram')
# plt.legend()
# plt.show()

## 6. SHAP Explanations

In [ ]:
# shap_results = compute_shap_explanations(
#     best_model, test_df[feature_cols], feature_names=feature_cols
# )
# print('Top 10 most important features (global):')
# for name, importance in shap_results['global_importance'][:10]:
#     print(f'  {name}: {importance:.4f}')
# 
# # SHAP summary plot
# import shap
# shap.summary_plot(shap_results['shap_values'], test_df[feature_cols][:200])

## 7. Gate Check

In [ ]:
# Full evaluation with fairness
# eval_results = evaluate_model(
#     y_true=test_df['outcome_binary'].values,
#     y_probs=test_probs_cal,
# )
# 
# passed, report = check_gates(eval_results)
# print(report)

# all_results = {
#     **baseline_results,
#     'random_forest': rf_metrics,
#     'xgboost_tabular': metrics_tab,
# }
# if 'metrics_text' in dir():
#     all_results['xgboost_text'] = metrics_text
# 
# comparison = pd.DataFrame(all_results).T
# print('\n=== Model Comparison (Literature-Aligned) ===')
# print(comparison[['test_accuracy', 'test_auc_roc', 'test_brier']].to_string())
# print(f'\nGates: accuracy >= {settings.model.min_test_accuracy}, '
#       f'AUC >= {settings.model.min_test_auc_roc}, '
#       f'Brier <= {settings.model.max_brier_score}')
# print(f'(Based on: Katz 70.2%, Aletras 79%, JES 2024 72%, PILOT 0.83 AUC)')

In [ ]:
# all_results = {
#     **baseline_results,
#     'xgboost_tabular': metrics_tab,
# }
# if 'metrics_text' in dir():
#     all_results['xgboost_text'] = metrics_text
# 
# comparison = pd.DataFrame(all_results).T
# print(comparison[['test_accuracy', 'test_auc_roc', 'test_brier']].to_string())